In [1]:
from pycirclize import Circos
import pandas as pd

In [2]:
names = [
    '43_A03590E1G4_WT202403310064',
 'B03607C4E6_WT2024071214941',
    '12_B03605F3G5_WT202403310048',
    '13_B03612A1C3_WT202403310056',
    '14_A03591A1C3_WT202403310045',
    '16_A03592A4C6_WT202403310044',
    '18_B03602C4D6_WT202405020031',
    '20_B03606F3G5_WT202405020032',
    '22_B03606C4E6_WT202403310050',
    '23_B03609A4D6_WT202404150263',
    '27_B03610C1E3_WT202403310051',
    '31_B03619A1D3_WT202403310052',
    '35_B03619E4G6_WT202403310053',
    '39_A03589A1D4_WT202403310046',
    '47_A03593C1F3_WT202403310068',
    
]
# cell_colors = {
#   '1' : '#9b38e9',
#   '2' : '#a89630',
#   '3': '#5b798b',
#   '4' : '#cb2505',
#   '5' : '#62e7dd',
#   '6' : '#245200',
#   '7' : '#374898',
#   '8' : '#6d85c7',
#   '9' : '#35c498',
#   '10' : '#9e2dc6',
#   '11' : '#2d7476',
#   '12' : '#cb0d6c',
#   '13' : '#20ea38',
#   '14' : '#0fabb6',
#   '15' : '#a59099',
#   '16' : '#2bea3a',
#   '17' : '#17b064',
#   '18' : '#52b8d5',
#   '19' : '#da2ef2',
#   '20' : '#6240f7',
#   '21' : '#c47233',
#   '22':'#a83b23',
#   '23':'#9994da'
# }
dic = {
    'hip_sc_1': 'hip_sc_18',
    'hip_sc_2': 'hip_sc_19',
    'hip_sc_3': 'hip_sc_20',
    'hip_sc_4': 'hip_sc_21',
    'hip_sc_5': 'hip_sc_22',
    'hip_sc_6': 'ctx_sc_1',
    'hip_sc_7': 'ctx_sc_2',
    'hip_sc_8': 'ctx_sc_3',
    'hip_sc_9': 'ctx_sc_4',
    'hip_sc_10': 'ctx_sc_5',
    'hip_sc_11': 'ctx_sc_6',
    'hip_sc_12': 'ctx_sc_7',
    'hip_sc_13': 'ctx_sc_8',
    'hip_sc_14': 'ctx_sc_9',
    'hip_sc_15': 'ctx_sc_10',
    'hip_sc_16': 'ctx_sc_11',
    'hip_sc_17': 'ctx_sc_12',
    'hip_sc_18': 'ctx_sc_13',
    'hip_sc_19': 'ctx_sc_14',
    'hip_sc_20': 'ctx_sc_15',
    'hip_sc_21': 'ctx_sc_16',
    'hip_sc_22': 'hip_sc_23',
    'hip_sc_23': 'ctx_sc_17',
}


cell_colors = {
'1' : '#374898',
  '2' : '#6d85c7',
  '3' : '#35c498',
  '4' : '#9e2dc6',
  '5' : '#2d7476',
  '6' : '#cb0d6c',
  '7' : '#20ea38',
  '8' : '#0fabb6',
  '9' : '#a59099',
  '10' : '#2bea3a',
  '11' : '#17b064',
  '12' : '#52b8d5',
  '13' : '#da2ef2',
  '14' : '#6240f7',
  '15' : '#c47233',
  '16':'#a83b23',
  '17':'#9994da',
  '18' : '#9b38e9',
  '19' : '#a89630',
  '20': '#5b798b',
  '21' : '#cb2505',
  '22' : '#62e7dd',
  '23' : '#245200',
}

for name in names:
    true_name = name.replace('.h5ad', '')
    csv = pd.read_csv(f'/data/work/05.cluster/FuseMap/0116/cci/{true_name}.csv')
    csv = csv[csv['is_significant'] == True]
    csv = csv[csv['lr_co_exp_ratio_pvalue'] < 0.01]
    
    # csv['SourceID'] = [i.split('-')[0].replace('hip_sc_', '') for i in csv['sr_pair']]
    
    # csv['DestinationID'] = [i.split('-')[1].replace('hip_sc_', '') for i in csv['sr_pair']]
    
    csv['SourceID'] = [dic[i.split('-')[0]].split('_')[-1] for i in csv['sr_pair']]
    csv['DestinationID'] = [dic[i.split('-')[1]].split('_')[-1] for i in csv['sr_pair']]
    
    csv['Ligand'] = [i.split('-')[0] for i in csv['lr_pair']]
    csv['Receptor'] = [i.split('-')[1] for i in csv['lr_pair']]
    csv['select'] = [True if 'CD' in i else False for i in csv['Ligand']]
    csv = csv[csv['select'] == False]
    csv['select'] = [True if 'CD' in i else False for i in csv['Receptor']]
    csv = csv[csv['select'] == False]
    # links = csv.groupby(['SourceGene', 'TargetGene', 'SourceID', 'DestinationID'], as_index=False)['lr_co_exp_num'].sum()

    # links.columns = ['source_gene', 'target_gene', 'source_cell', 'target_cell', 'value']
    links = csv.groupby(['SourceID', 'DestinationID'], as_index=False)['lr_co_exp_num'].sum()

    matrix = links.pivot(index='SourceID', columns='DestinationID', values='lr_co_exp_num')

    # 填充缺失值为 0（如果某些组合不存在）
    matrix = matrix.fillna(0)

    def link_kws_handler(from_label: str, to_label: str):
        if from_label in ("18") or from_label in ("19") or to_label in ("18") or to_label in ("19"):
            # Set alpha, zorder values higher than other links for highlighting
            return dict(alpha=0.5, zorder=1.0)
        else:
            return dict(alpha=0.1, zorder=0)

    # Initialize Circos instance for chord diagram plot
    circos = Circos.chord_diagram(
        matrix,
        space=2,
        cmap=cell_colors,
        label_kws=dict(size=12),
        link_kws=dict(direction=1, ec="black", lw=0.5),
        link_kws_handler=link_kws_handler,
    )

    # fig = circos.plotfig()
    circos.savefig(f"/data/work/05.cluster/FuseMap/0116/cciPlot/circle/{true_name}.png")

In [3]:
csv

,Unnamed: 0,from,to,pathway,type,lr_pair,lr_product,lr_co_exp_num,lr_co_exp_ratio,lr_co_exp_ratio_pvalue,is_significant,sr_pair,SourceID,DestinationID,Ligand,Receptor,select
0,2185,NTS,NTSR1,NTS,Secreted Signaling,NTS-NTSR1,0.018149,20,0.005951,0.0,True,hip_sc_1-hip_sc_2,18,19,NTS,NTSR1,False
1,2310,SEMA3C,NRP1,SEMA3,Secreted Signaling,SEMA3C-NRP1,0.028002,21,0.013068,0.0,True,hip_sc_1-hip_sc_3,18,20,SEMA3C,NRP1,False
2,2286,SEMA3A,NRP1,SEMA3,Secreted Signaling,SEMA3A-NRP1,0.028625,13,0.008090,0.0,True,hip_sc_1-hip_sc_3,18,20,SEMA3A,NRP1,False
3,2185,NTS,NTSR1,NTS,Secreted Signaling,NTS-NTSR1,0.040435,42,0.013064,0.0,True,hip_sc_2-hip_sc_1,19,18,NTS,NTSR1,False
4,2185,NTS,NTSR1,NTS,Secreted Signaling,NTS-NTSR1,0.022260,13,0.011130,0.0,True,hip_sc_2-hip_sc_12,19,7,NTS,NTSR1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,3244,COL1A2,ITGB8,COLLAGEN,ECM-Receptor,COL1A2-ITGB8,0.007438,12,0.005579,0.0,True,hip_sc_22-hip_sc_23,23,17,COL1A2,ITGB8,False
85,2571,COL2A1,ITGA2,COLLAGEN,ECM-Receptor,COL2A1-ITGA2,0.005114,6,0.002789,0.0,True,hip_sc_22-hip_sc_23,23,17,COL2A1,ITGA2,False
86,3775,DLL1,NOTCH1,NOTCH,Cell-Cell Contact,DLL1-NOTCH1,0.008790,9,0.004654,0.0,True,hip_sc_23-hip_sc_21,17,16,DLL1,NOTCH1,False
87,2586,COL4A5,ITGA2,COLLAGEN,ECM-Receptor,COL4A5-ITGA2,0.010341,7,0.003619,0.0,True,hip_sc_23-hip_sc_21,17,16,COL4A5,ITGA2,False


In [4]:
names = [
    '43_A03590E1G4_WT202403310064',
 'B03607C4E6_WT2024071214941',
    '12_B03605F3G5_WT202403310048',
    '13_B03612A1C3_WT202403310056',
    '14_A03591A1C3_WT202403310045',
    '16_A03592A4C6_WT202403310044',
    '18_B03602C4D6_WT202405020031',
    '20_B03606F3G5_WT202405020032',
    '22_B03606C4E6_WT202403310050',
    '23_B03609A4D6_WT202404150263',
    '27_B03610C1E3_WT202403310051',
    '31_B03619A1D3_WT202403310052',
    '35_B03619E4G6_WT202403310053',
    '39_A03589A1D4_WT202403310046',
    '47_A03593C1F3_WT202403310068',
    
]
# cell_colors = {
#   '1' : '#9b38e9',
#   '2' : '#a89630',
#   '3': '#5b798b',
#   '4' : '#cb2505',
#   '5' : '#62e7dd',
#   '6' : '#245200',
#   '7' : '#374898',
#   '8' : '#6d85c7',
#   '9' : '#35c498',
#   '10' : '#9e2dc6',
#   '11' : '#2d7476',
#   '12' : '#cb0d6c',
#   '13' : '#20ea38',
#   '14' : '#0fabb6',
#   '15' : '#a59099',
#   '16' : '#2bea3a',
#   '17' : '#17b064',
#   '18' : '#52b8d5',
#   '19' : '#da2ef2',
#   '20' : '#6240f7',
#   '21' : '#c47233',
#   '22':'#a83b23',
#   '23':'#9994da'
# }

cell_colors = {
'1' : '#374898',
  '2' : '#6d85c7',
  '3' : '#35c498',
  '4' : '#9e2dc6',
  '5' : '#2d7476',
  '6' : '#cb0d6c',
  '7' : '#20ea38',
  '8' : '#0fabb6',
  '9' : '#a59099',
  '10' : '#2bea3a',
  '11' : '#17b064',
  '12' : '#52b8d5',
  '13' : '#da2ef2',
  '14' : '#6240f7',
  '15' : '#c47233',
  '16':'#a83b23',
  '17':'#9994da',
  '18' : '#9b38e9',
  '19' : '#a89630',
  '20': '#5b798b',
  '21' : '#cb2505',
  '22' : '#62e7dd',
  '23' : '#245200',
}



for name in names:
    true_name = name.replace('.h5ad', '')
    csv = pd.read_csv(f'/data/work/05.cluster/FuseMap/0116/cci/{true_name}.csv')
    csv = csv[csv['is_significant'] == True]
    csv = csv[csv['lr_co_exp_ratio_pvalue'] < 0.01]
    # csv['SourceID'] = [i.split('-')[0].replace('hip_sc_', '') for i in csv['sr_pair']]
    
    # csv['DestinationID'] = [i.split('-')[1].replace('hip_sc_', '') for i in csv['sr_pair']]
    csv['SourceID'] = [dic[i.split('-')[0]].split('_')[-1] for i in csv['sr_pair']]
    csv['DestinationID'] = [dic[i.split('-')[1]].split('_')[-1] for i in csv['sr_pair']]
    
    csv['Ligand'] = [i.split('-')[0] for i in csv['lr_pair']]
    csv['Receptor'] = [i.split('-')[1] for i in csv['lr_pair']]
    csv['select'] = [True if 'CD' in i else False for i in csv['Ligand']]
    csv = csv[csv['select'] == False]
    csv['select'] = [True if 'CD' in i else False for i in csv['Receptor']]
    csv = csv[csv['select'] == False]
    # links = csv.groupby(['SourceGene', 'TargetGene', 'SourceID', 'DestinationID'], as_index=False)['lr_co_exp_num'].sum()

    # links.columns = ['source_gene', 'target_gene', 'source_cell', 'target_cell', 'value']
    links = csv.groupby(['SourceID', 'DestinationID'], as_index=False)['lr_co_exp_num'].sum()

    matrix = links.pivot(index='SourceID', columns='DestinationID', values='lr_co_exp_num')

    # 填充缺失值为 0（如果某些组合不存在）
    matrix = matrix.fillna(0)

    def link_kws_handler(from_label: str, to_label: str):
        # if from_label in ("1") or from_label in ("2") or to_label in ("2") or to_label in ("1"):
            # Set alpha, zorder values higher than other links for highlighting
            return dict(alpha=0.5, zorder=1.0)
        # else:
            # return dict(alpha=0.1, zorder=0)

    # Initialize Circos instance for chord diagram plot
    circos = Circos.chord_diagram(
        matrix,
        space=2,
        cmap=cell_colors,
        label_kws=dict(size=12),
        link_kws=dict(direction=1, ec="black", lw=0.5),
        link_kws_handler=link_kws_handler,
    )

    # fig = circos.plotfig()
    circos.savefig(f"/data/work/05.cluster/FuseMap/0116/cciPlot/circle/{true_name}_highlight_all.png")

In [5]:
1

1

In [6]:
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [11]:
for i in [
    '43_A03590E1G4_WT202403310064',
 'B03607C4E6_WT2024071214941',
    '12_B03605F3G5_WT202403310048',
    '13_B03612A1C3_WT202403310056',
    '14_A03591A1C3_WT202403310045',
    '16_A03592A4C6_WT202403310044',
    '18_B03602C4D6_WT202405020031',
    '20_B03606F3G5_WT202405020032',
    '22_B03606C4E6_WT202403310050',
    '23_B03609A4D6_WT202404150263',
    '27_B03610C1E3_WT202403310051',
    '31_B03619A1D3_WT202403310052',
    '35_B03619E4G6_WT202403310053',
    '39_A03589A1D4_WT202403310046',
    '47_A03593C1F3_WT202403310068',
    
]:
    
    csv = pd.read_csv(f'/data/work/05.cluster/FuseMap/0116/cci/{i}.csv', index_col = 'Unnamed: 0')
    csv = csv[csv['lr_pair'].isin([
        'NTS-NTSR1',
        'SEMA3C-NRP1',
        'SEMA3A-NRP1',
        'MIF-CXCR4',
        'NRXN1-NLGN1'
    ])]
    
    scaler = MinMaxScaler()
    csv['lr_co_exp_ratio_scaled'] = scaler.fit_transform(csv[['lr_co_exp_ratio']])
    csv['sr_pair'] = [dic[i.split('-')[0]] + '-' + dic[i.split('-')[1]] for i in csv['sr_pair']]
    # csv['lr_pair'] = [dic[i.split('-')[0]] + '-' + dic[i.split('-')[1]] for i in csv['lr_pair']]
        
    plt.figure(figsize=(len(list(set(csv['sr_pair'])))/3, len(list(set(csv['lr_pair'])))/4))
    sns.scatterplot(x='sr_pair', 
                    y='lr_pair', 
                    size='lr_co_exp_ratio', 
                    hue = 'lr_co_exp_ratio',
                    data=csv,
                    palette = 'viridis',
                    # legend=False  # 先关闭legend，稍后统一处理
                   )
    handles, labels = plt.gca().get_legend_handles_labels()
    labels = [f"{float(label):.2f}" for label in labels] 
    plt.legend(handles, labels, bbox_to_anchor=(1.05, 1), loc='upper left')
    significant_points = csv[csv['is_significant'] == True]
    plt.scatter(x=significant_points['sr_pair'], 
                y=significant_points['lr_pair'], 
                facecolors='none', 
                edgecolors='red', 
                s=100)  # 调整s参数以控制圆圈大小

    plt.xlim(-1, len(list(set(csv['sr_pair'])))+1)
    # plt.margins(y=0.1)  
    plt.xticks(rotation=45, ha = 'right')
    plt.xlabel(xlabel = None)
    plt.ylabel(ylabel = None)
    plt.title(f'')
    plt.tight_layout()
    plt.savefig(f'/data/work/05.cluster/FuseMap/0116/cciPlot/dotplot_sub/dot_plot_{i}.pdf', bbox_inches = 'tight')
    plt.close()
    # plt.show()
    # break
    

/tmp/ipykernel_124/1749430090.py:59: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()
/tmp/ipykernel_124/1749430090.py:59: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()
/tmp/ipykernel_124/1749430090.py:59: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()
/tmp/ipykernel_124/1749430090.py:59: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()
/tmp/ipykernel_124/1749430090.py:59: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()
/tmp/ipykernel_124/1749430090.py:59: UserWarning: Tight

In [ ]:
1